# Observation → Karar: Uçtan Uca (Edge Policy / v10 Remastered)

Bu defter, **tek bir tur** için verinin nereden geldiğini ve **ReworkedEdgePolicy**'nin nasıl `[[planet_id, açı, gemi], …]` ürettiğini adım adım anlatır.

**Kapsam:** `obs_builder` → `reworked_pipeline` → `reworked_model` (GNN + gate/frac) → `reworked_rollout` (mask, env launch) → buffer / ödül / PPO.

**Eğitim stack'i:** `reworked_train_remastered` + `ReworkedEdgePolicy` + kampanya credit (`reworked_remastered_reward`).

Detaylı feature formülleri: `docs/reworked_features_reference.md`

## Mimari (tek tur)

```
kaggle env.observation (ham dict)
    │  planets[], fleets[], step, angular_velocity …
    ▼
obs_builder.build_observation(player)
    │  planet_feats (34-D)  planet_mask  global_feats (14-D)
    │  edge_bias (T×T×6)    fleet routing slot arrays
    ▼
reworked_pipeline
    │  board_state_from_obs() → owner, ships, dist …
    │  build_reworked_features() → node (16-D), edge (4-D)
    │  pack_reworked_batch() → torch (B,P,·)
    ▼
ReworkedPlanetGraphEncoder (3× DenseGINE)
    │  node_tokens (B,P,d)  edge_tokens (B,P,P,d)  graph_summary (B,d)
    ▼
ReworkedEdgePolicy.act()
    │  gate[h,t] ~ Bernoulli   frac[h,t] ~ Beta   (her kenar bağımsız)
    ▼
edge_action_to_env() + navigation.compute_launch()
    │  [[src_id, angle, ships], …]
    ▼
env.step()
    │
    ▼ (episode sonu)
attribute_remastered_rewards → credit_target[t]
compute_gae → adv_target[t] = credit_target[t]  (target_valid maskeli)
ppo_update → per-target clipped surrogate
```

## 0) Kurulum

Notebook'u `Orbit_Wars/docs/` içinden açsanız da olur — kurulum hücresi üst dizinlere bakarak `obs_builder.py` olan `Orbit_Wars/` kökünü bulur ve `chdir` yapar.

In [62]:
import os, sys
from pathlib import Path

# Notebook docs/ içinden açılsa bile Orbit_Wars paket kökünü bul.
OW = None
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "obs_builder.py").exists():
        OW = base
        break
    if (base / "Orbit_Wars" / "obs_builder.py").exists():
        OW = base / "Orbit_Wars"
        break
if OW is None:
    raise RuntimeError(
        "obs_builder.py bulunamadı. cwd=%s — notebook'u Orbit_Wars/ veya "
        "Orbit_Wars/docs/ altından açın." % Path.cwd()
    )

os.chdir(OW)
if str(OW) not in sys.path:
    sys.path.insert(0, str(OW))

# Kaynak dosyaları düzenledikten sonra kernel restart etmeden güncel kodu yükle.
for _m in list(sys.modules):
    if _m.startswith(("reworked_", "obs_builder", "feature_extraction", "navigation")):
        del sys.modules[_m]

import numpy as np
import torch
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
print("OW =", OW.resolve())

OW = /Users/doruksarigun/Kaggle/orbit-wars/Orbit_Wars


## 1) Ham env gözlemi

`env.step()` sonrası her oyuncu için dönen `observation` bir **dict**:

| Alan | Tip | Anlam |
|------|-----|-------|
| `planets` | `list[list]` | `[id, owner, x, y, prod, ships, radius]` |
| `fleets` | `list[list]` | `[id, owner, x, y, angle, src_id, ships]` |
| `step` | int | Oyun turu |
| `angular_velocity` | float | Dönen gezegenler için ω |

Policy bu ham dict'i **doğrudan görmez** — önce `build_observation` sabit boyutlu tensörlere çevirir.

In [63]:
from kaggle_environments import make

PLAYER = 0          # eğitilen agent (P0)
P_MAX = 44          # reworked_rollout.P_MAX ile aynı
F_MAX = 64
MAX_STEPS = 500
SEED = 0

env = make("orbit_wars", debug=False)
env.configuration["randomSeed"] = SEED
env.reset(num_agents=2)
state = env.step([[], []])  # ilk boş hamle

raw = state[PLAYER]["observation"]
print("step =", raw.get("step"))
print("n_planets =", len(raw["planets"]))
print("n_fleets =", len(raw.get("fleets") or []))
print("ilk gezegen:", raw["planets"][0])
print("sütunlar: [id, owner, x, y, production, ships, radius]")

step = 1
n_planets = 28
n_fleets = 0
ilk gezegen: [0, -1, 75.73098307528582, 96.87644954593827, 2.09861228866811, 63, 3]
sütunlar: [id, owner, x, y, production, ships, radius]


## 2) `build_observation` — sabit şekilli `Observation`

Girdi: ham dict + **bakış açısı** (`player`).

Çıktı (`obs_builder.Observation`):

| Alan | Şekil | Rol |
|------|-------|-----|
| `planet_feats` | `(P_max, 34)` | Gezegen özellikleri (normalize) |
| `planet_mask` | `(P_max,)` | Gerçek gezegen = True |
| `global_feats` | `(14,)` | Tur / ekonomi özeti |
| `edge_bias` | `(T,T,6)` | Çiftler arası geometri (T=P+F+1) |
| `fleet_*_slot` | filo routing | Uçuşta filoların slot pointer'ları |
| `planet_ids` | `(P_max,)` | slot → env planet id |
| `source_mask` | `(P_max,)` | launch kaynağı olabilir mi |

**Permütasyon:** Gezegen sırası env'den gelir; GNN slot-index kullanır, `planet_ids` env id eşlemesi için saklanır.

In [64]:
from obs_builder import build_observation, PLANET_FEATURE_DIM, GLOBAL_FEATURE_DIM
from feature_extraction import parse_planets

av = float(raw.get("angular_velocity", 0.0) or 0.0)
turn = int(raw.get("step", 0))

obs = build_observation(
    raw, player=PLAYER, angular_velocity=av,
    p_max=P_MAX, f_max=F_MAX, turn=turn, max_turns=MAX_STEPS,
)

n = int(obs.planet_mask.sum())
print(f"gerçek gezegen sayısı n = {n}")
print(f"planet_feats: {obs.planet_feats.shape}  (beklenen: ({P_MAX}, {PLANET_FEATURE_DIM}))")
print(f"global_feats: {obs.global_feats.shape}")
print(f"edge_bias:    {obs.edge_bias.shape}")
print(f"planet_ids[:n] = {obs.planet_ids[:n]}")

gerçek gezegen sayısı n = 28
planet_feats: (44, 34)  (beklenen: (44, 34))
global_feats: (14,)
edge_bias:    (109, 109, 6)
planet_ids[:n] = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27]


### 2a) `planet_feats` — 34 kanal (obs_builder)

İndeks sabitleri `obs_builder.py` içinde (`PF_IDX_*`). Özet:

| İndeks | İsim | Formül / anlam |
|--------|------|----------------|
| 0 | owner_self | 1 ise benim gezegenim |
| 1–3 | owner_enemy_j | Düşman slotları (perspektif-göreli) |
| 4 | owner_neutral | Nötr |
| 5 | ships | `ships / 100` |
| 6 | production | `prod / 5` |
| 7 | radius | `radius / 5` |
| 8–10 | r_sun, cosθ, sinθ | Güneş merkezli polar |
| 11–14 | static / rotating / comet / ω | Tip bayrakları |
| 15 | log_ships | `log(s+1)/log(1000)` |
| 16–23 | incoming buckets | ETA dilimlerinde gelen filolar |
| 24–31 | incoming özet | toplam, min ETA, tehdit |
| 32–33 | capture_value, defense_deficit | legacy skorlar |

Reworked stack bu 34-D bloğu **yeniden türetmez** — de-normalize edip üzerine 16-D node feature kurar.

In [65]:
PF_NAMES = [
    "owner_self", "enemy_0", "enemy_1", "enemy_2", "neutral",
    "ships/100", "prod/5", "radius/5", "r_sun/50", "cos_theta", "sin_theta",
    "is_static", "is_rotating", "is_comet", "omega",
    "log_ships", "inc_self_b0", "inc_self_b1", "inc_self_b2", "inc_self_b3",
    "inc_enemy_b0", "inc_enemy_b1", "inc_enemy_b2", "inc_enemy_b3",
    "inc_self_tot", "inc_enemy_tot", "inc_self_min_eta", "inc_enemy_min_eta",
    "enemy_soonest", "enemy_largest", "enemy_largest_eta", "net_threat",
    "capture_value", "defense_deficit",
]
pf = obs.planet_feats[:n]
df_pf = pd.DataFrame(pf, columns=PF_NAMES)
df_pf.insert(0, "slot", range(n))
df_pf.insert(1, "planet_id", obs.planet_ids[:n])
display(df_pf.round(3).head(8))

,slot,planet_id,owner_self,enemy_0,enemy_1,enemy_2,neutral,ships/100,prod/5,radius/5,...,inc_self_tot,inc_enemy_tot,inc_self_min_eta,inc_enemy_min_eta,enemy_soonest,enemy_largest,enemy_largest_eta,net_threat,capture_value,defense_deficit
0,0,0,0.0,0.0,0.0,0.0,1.0,0.63,0.6,0.420,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06,0.0
1,1,1,0.0,0.0,0.0,0.0,1.0,0.63,0.6,0.420,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06,0.0
2,2,2,0.0,0.0,0.0,0.0,1.0,0.63,0.6,0.420,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06,0.0
3,3,3,0.0,0.0,0.0,0.0,1.0,0.63,0.6,0.420,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06,0.0
4,4,4,0.0,0.0,0.0,0.0,1.0,0.49,0.8,0.477,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.08,0.0
5,5,5,0.0,0.0,0.0,0.0,1.0,0.49,0.8,0.477,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.08,0.0
6,6,6,0.0,0.0,0.0,0.0,1.0,0.49,0.8,0.477,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.08,0.0
7,7,7,0.0,0.0,0.0,0.0,1.0,0.49,0.8,0.477,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.08,0.0


### 2b) `global_feats` — 14 kanal

Tur oranı, ω, gezegen sayıları, toplam gemi/üretim, filo sayısı, hava üstünlüğü vb. Encoder'da `graph_summary`'ye katkı verir.

In [66]:
GF_NAMES = [
    "turn/200", "omega", "n_my/25", "n_enemy/25", "n_neutral/25",
    "my_ships/100", "enemy_ships/100", "my_prod/50", "enemy_prod/50",
    "n_fleets/25", "incoming_other/100", "incoming_self/100",
    "my_capture_value/5000", "air_advantage",
]
display(pd.DataFrame([obs.global_feats], columns=GF_NAMES).round(4))

,turn/200,omega,n_my/25,n_enemy/25,n_neutral/25,my_ships/100,enemy_ships/100,my_prod/50,enemy_prod/50,n_fleets/25,incoming_other/100,incoming_self/100,my_capture_value/5000,air_advantage
0,0.005,0.0358,0.04,0.04,1.04,0.1,0.1,0.02,0.02,0.0,0.0,0.0,0.0002,0.0


## 3) `reworked_pipeline` — BoardState + 16-D node + 4-D edge

### 3a) De-normalizasyon → `BoardState`

`board_state_from_obs`:
- `ships = planet_feats[:,5] × 100`
- `production = planet_feats[:,6] × 5`
- `distance_matrix = edge_bias[:n,:n,0]` (normalize mesafe)
- `owner = effective_owner_from_obs(...)` — one-hot'tan sentetik owner id

Allocator ve feature fonksiyonları **aynı BoardState**'i görür.

In [67]:
from reworked_pipeline import board_state_from_obs, build_reworked_features
from reworked_model import REWORKED_NODE_FEATURE_DIM, REWORKED_EDGE_FEATURE_DIM

state = board_state_from_obs(obs, PLAYER, remaining_turns=MAX_STEPS - turn)
bundle = build_reworked_features(obs, PLAYER, remaining_turns=MAX_STEPS - turn)

print("BoardState:")
print(f"  n={state.n}  player={state.player}  remaining_turns={state.remaining_turns}")
print(f"  owner[:6] = {state.owner[:6]}")
print(f"  ships[:6] = {state.ships[:6].round(1)}")
print(f"node_feats: {bundle['node_feats'].shape}  (beklenen: ({state.n}, {REWORKED_NODE_FEATURE_DIM}))")
print(f"edge_feats: {bundle['edge_feats'].shape}")

BoardState:
  n=28  player=0  remaining_turns=499
  owner[:6] = [-1 -1 -1 -1 -1 -1]
  ships[:6] = [63. 63. 63. 63. 49. 49.]
node_feats: (28, 16)  (beklenen: (28, 16))
edge_feats: (28, 28, 4)


### 3b) Node feature — 16 kanal (`reworked_features`)

| # | Kanal | Kaynak |
|---|-------|--------|
| 0–9 | `standard_node_features` | owner one-hot (5) + log_ships + prod + radius + flags |
| 10 | `discounted_production_potential` | Kalan turda indirimli üretim değeri (dpp) |
| 11 | `closeness_centrality` | Mesafe tabanlı merkezilik |
| 12 | `potential_threat` | Düşman garnizon baskısı (hız-duyarlı ETA) |
| 13 | `potential_support` | Dost takviye kapasitesi |
| 14 | `friendly_compactness` | Kendi topraklarında iç/dış hat |
| 15 | `capture_affordability` | `reach − demand` (bu hedefe yeterli güç var mı?) |

**Önemli:** `capture_affordability` **hedef node**'da; gate kararı **kenar (h→t)** başına ayrı.

In [68]:
NODE_NAMES = [
    "own_self", "enemy_0", "enemy_1", "enemy_2", "neutral",
    "log_ships", "prod/5", "radius/5", "rotating", "comet",
    "dpp", "closeness", "pot_threat", "pot_support", "friendly_compact", "capture_afford",
]
nf = bundle["node_feats"]
df_nf = pd.DataFrame(nf, columns=NODE_NAMES)
df_nf.insert(0, "slot", range(state.n))
df_nf.insert(1, "id", obs.planet_ids[:state.n])
df_nf.insert(2, "ships", state.ships.round(0).astype(int))
df_nf.insert(3, "owner", state.owner)
display(df_nf.round(3).head(10))

# 5 garnizonlu nötrler
weak = np.where((state.owner == -1) & (state.ships == 5))[0]
if len(weak):
    print("\n5 garnizonlu nötr slotlar:")
    display(df_nf.iloc[weak][["slot", "id", "ships", "capture_afford", "dpp"]].round(3))

,slot,id,ships,owner,own_self,enemy_0,enemy_1,enemy_2,neutral,log_ships,prod/5,radius/5,rotating,comet,dpp,closeness,pot_threat,pot_support,friendly_compact,capture_afford
0,0,0,63,-1,0.0,0.0,0.0,0.0,1.0,0.602,0.6,0.420,0.0,0.0,0.596,1.525,0.004,0.024,0.0,-1.08
1,1,1,63,-1,0.0,0.0,0.0,0.0,1.0,0.602,0.6,0.420,0.0,0.0,0.596,1.525,0.004,0.006,0.0,-1.08
2,2,2,63,-1,0.0,0.0,0.0,0.0,1.0,0.602,0.6,0.420,0.0,0.0,0.596,1.525,0.006,0.004,0.0,-1.08
3,3,3,63,-1,0.0,0.0,0.0,0.0,1.0,0.602,0.6,0.420,0.0,0.0,0.596,1.525,0.024,0.004,0.0,-1.08
4,4,4,49,-1,0.0,0.0,0.0,0.0,1.0,0.566,0.8,0.477,0.0,0.0,0.795,1.639,0.004,0.028,0.0,-0.80
5,5,5,49,-1,0.0,0.0,0.0,0.0,1.0,0.566,0.8,0.477,0.0,0.0,0.795,1.639,0.006,0.005,0.0,-0.80
6,6,6,49,-1,0.0,0.0,0.0,0.0,1.0,0.566,0.8,0.477,0.0,0.0,0.795,1.639,0.005,0.006,0.0,-0.80
7,7,7,49,-1,0.0,0.0,0.0,0.0,1.0,0.566,0.8,0.477,0.0,0.0,0.795,1.639,0.028,0.004,0.0,-0.80
8,8,8,28,-1,0.0,0.0,0.0,0.0,1.0,0.487,0.8,0.477,0.0,0.0,0.795,1.643,0.004,0.012,0.0,-0.38
9,9,9,28,-1,0.0,0.0,0.0,0.0,1.0,0.487,0.8,0.477,0.0,0.0,0.795,1.643,0.007,0.004,0.0,-0.38


### 3c) Edge feature — 4 kanal (yönlü h→t)

| # | Kanal | Anlam |
|---|-------|-------|
| 0 | distance | Normalize mesafe |
| 1 | crosses_sun | Güneş geçişi (0/1) |
| 2 | friendly_flux | Uçuşta dost filo akışı (kaynak→hedef) |
| 3 | enemy_flux | Uçuşta düşman filo akışı |

In [69]:
ef = bundle["edge_feats"]
h, t = 0, 1  # örnek kenar
print(f"edge[h={h} -> t={t}] (id {obs.planet_ids[h]} -> {obs.planet_ids[t]}):")
for i, name in enumerate(["dist", "cross_sun", "friendly_flux", "enemy_flux"]):
    print(f"  {name}: {ef[h,t,i]:.4f}")

edge[h=0 -> t=1] (id 0 -> 1):
  dist: 0.5146
  cross_sun: 0.0000
  friendly_flux: 0.0000
  enemy_flux: 0.0000


### 3d) `pack_reworked_batch` — PyTorch tensörleri

Farklı `n` değerleri sağ-padding ile `P_max`'a tamamlanır; `planet_mask=False` olan slotlar GNN'de sıfırlanır.

In [70]:
from reworked_pipeline import pack_reworked_batch

device = torch.device("cpu")
batch = pack_reworked_batch([(obs, PLAYER)], device=device, p_max=P_MAX)
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"{k:14s} {tuple(v.shape)}  {v.dtype}")
    else:
        print(f"{k:14s} list len={len(v)}")

node_feats     (1, 44, 16)  torch.float32
edge_feats     (1, 44, 44, 4)  torch.float32
global_feats   (1, 14)  torch.float32
planet_mask    (1, 44)  torch.bool
states         list len=1


## 4) GNN Encoder (`ReworkedPlanetGraphEncoder`)

1. `node_embed`, `edge_embed`, `global_embed` → `d_model` (varsayılan 128)
2. `n_layers` adet **DenseGINE** mesaj geçişi (tam graf, maskeli)
3. Maskeli mean/max pool + global → `graph_summary`

Çıktılar edge policy head'lerinin girdisi.

In [71]:
from reworked_model import ReworkedPlanetGraphEncoder

enc = ReworkedPlanetGraphEncoder(d_model=32, n_layers=2)  # demo: küçük
with torch.no_grad():
    enc_out = enc(
        batch["node_feats"], batch["edge_feats"],
        batch["global_feats"], batch["planet_mask"],
    )
for k, v in enc_out.items():
    print(f"{k}: {tuple(v.shape)}")

node_tokens: (1, 44, 32)
edge_tokens: (1, 44, 44, 32)
graph_summary: (1, 32)


## 5) Maskeler — hangi kenarlar / hedefler geçerli?

### `edge_mask[h,t]` (`_edge_valid_mask`)
- `h` benim gezegenim ve `ships[h] >= min_send`
- `t != h` (herhangi bir başka gezegen)
- Saldırı ve savunma kenarları **birlikte** (yön belirgin)

### `target_valid[t]` (`_edge_target_valid_mask`) — **PPO'ya giren hedefler**
- `gate[h,t] > 0.5` olan en az bir kenar **veya** o tura launch gitti
- Ulaşılabilir ama gate kapalı hedefler **dahil değil**

In [72]:
import importlib
import reworked_rollout
importlib.reload(reworked_rollout)
from reworked_rollout import _edge_valid_mask, _edge_target_valid_mask, MIN_SEND
print("reworked_rollout:", reworked_rollout.__file__)

st0 = batch["states"][0]
edge_np = _edge_valid_mask(st0, P_MAX, PLAYER, min_send=MIN_SEND)
n_active_targets_geom = int(edge_np.any(axis=0)[:st0.n].sum())
print(f"Geometrik ulaşılabilir hedef sayısı: {n_active_targets_geom}")
print(f"Kendi kaynak gezegen slotları: {np.where(st0.owner == PLAYER)[0]}")

reworked_rollout: /Users/doruksarigun/Kaggle/orbit-wars/Orbit_Wars/reworked_rollout.py
Geometrik ulaşılabilir hedef sayısı: 27
Kendi kaynak gezegen slotları: [16]


## 6) `ReworkedEdgePolicy.act()` — gate + frac

Her geçerli kenar `(h→t)` için **bağımsız** iki karar:

| | Dağılım | Anlam |
|--|---------|-------|
| `gate[h,t]` | Bernoulli(sigmoid(logit)) | Bu kenardan filo gönder? |
| `frac[h,t]` | Beta(α,β) | Kaynak garnizonunun hangi oranı? |

**Gemi sayısı:** `floor(frac[h,t] × garrison[h])` (gate açıksa).

**Deterministic replay:** `gate = (prob > 0.5)`; hiç seçim yoksa en yüksek prob'lu tek kenar açılır.

**Log-prob:** hedef `t` bazında toplanır → `target_logp[t] = Σ_h log p(h→t)`.

In [73]:
from reworked_train import load_checkpoint

ckpt_path = OW / "checkpoints_remastered_v11" / "latest.pt"
if not ckpt_path.exists():
    ckpt_path = OW / "checkpoints_remastered_v10" / "latest.pt"
policy, ckpt = load_checkpoint(ckpt_path, device=device)
policy.eval()
print("checkpoint:", ckpt_path.name, "iter", ckpt.get("iter", "?"))

edge_t = torch.from_numpy(edge_np)[None].to(device)
with torch.no_grad():
    action, target_logp, value = policy.act(
        node_feats=batch["node_feats"], edge_feats=batch["edge_feats"],
        global_feats=batch["global_feats"], planet_mask=batch["planet_mask"],
        edge_mask=edge_t, deterministic=True,
    )
gate = action["gate"][0].cpu().numpy()[:st0.n, :st0.n]
frac = action["frac"][0].cpu().numpy()[:st0.n, :st0.n]
tv = _edge_target_valid_mask(gate, edge_np)
print(f"value V(s) = {float(value[0]):.4f}")
print(f"target_valid True count = {tv[:st0.n].sum()}  (geometrik: {n_active_targets_geom})")

checkpoint: latest.pt iter ?
value V(s) = 0.0866
target_valid True count = 1  (geometrik: 27)


In [82]:
gate

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0

In [74]:
# Tek kaynak gezegenimiz varsa: tüm hedeflere gate/frac tablosu
own_slots = [int(i) for i in range(st0.n) if st0.owner[i] == PLAYER]
ids = obs.planet_ids[:st0.n]
rows = []
for h in own_slots:
    garr = int(st0.ships[h])
    for t in range(st0.n):
        if t == h or not edge_np[h, t]:
            continue
        g = float(gate[h, t])
        f = float(frac[h, t])
        req = int(np.floor(garr * f)) if g > 0.5 else 0
        rows.append({
            "src_slot": h, "src_id": int(ids[h]), "garrison": garr,
            "tgt_slot": t, "tgt_id": int(ids[t]), "tgt_ships": int(st0.ships[t]),
            "tgt_owner": int(st0.owner[t]),
            "gate": round(g, 3), "frac": round(f, 3), "req_ships": req,
        })
df_edges = pd.DataFrame(rows)
if len(df_edges):
    display(df_edges.sort_values(["gate", "req_ships"], ascending=False).head(15))
    weak_rows = df_edges[(df_edges["tgt_ships"] == 5) & (df_edges["tgt_owner"] == -1)]
    if len(weak_rows):
        print("\n5 garnizonlu nötrlere ayırma:")
        display(weak_rows)
else:
    print("Bu turda geçerli kenar yok (min_send veya kaynak yok).")

,src_slot,src_id,garrison,tgt_slot,tgt_id,tgt_ships,tgt_owner,gate,frac,req_ships
17,16,16,10,18,18,6,-1,1.0,0.869,8
0,16,16,10,0,0,63,-1,0.0,0.869,0
1,16,16,10,1,1,63,-1,0.0,0.869,0
2,16,16,10,2,2,63,-1,0.0,0.869,0
3,16,16,10,3,3,63,-1,0.0,0.863,0
4,16,16,10,4,4,49,-1,0.0,0.869,0
5,16,16,10,5,5,49,-1,0.0,0.869,0
6,16,16,10,6,6,49,-1,0.0,0.869,0
7,16,16,10,7,7,49,-1,0.0,0.862,0
8,16,16,10,8,8,28,-1,0.0,0.869,0


## 7) `edge_action_to_env` — somut launch listesi

Adımlar (kaynak `h` için):
1. `gate[h,t]≤0.5` → atla
2. `req = floor(frac[h,t] × garrison[h])`
3. Toplam `req > garrison` → **ölçekle** (feasibility clamp)
4. `req < min_send` → atla
5. `navigation.compute_launch(src, tgt, ships)` → açı + ETA

Env formatı: `[planet_id, angle_radians, ships]`

In [75]:
from reworked_rollout import edge_action_to_env
from feature_extraction import parse_planets

parsed = parse_planets(raw)
row = {"gate": action["gate"][0].cpu().numpy(), "frac": action["frac"][0].cpu().numpy()}
env_action, launches = edge_action_to_env(
    st0, obs, PLAYER, row, parsed, av, min_send=MIN_SEND,
)

print(f"{len(launches)} launch, env_action:")
for ea, L in zip(env_action, launches):
    print(f"  env {ea}  |  slot {L.src}->{L.tgt}  ships={L.ships}  kind={L.kind}  angle={L.angle:.3f}")

tv2 = _edge_target_valid_mask(gate, edge_np, launches=launches)
print(f"target_valid after launches: {tv2[:st0.n].sum()}")

0 launch, env_action:
target_valid after launches: 1


## 8) Buffer kaydı — `EdgeTransition`

Her `(player, turn)` için **tek** transition; içinde tüm kenarların `gate/frac` matrisi saklanır.

| Alan | Şekil | Rol |
|------|-------|-----|
| `gate`, `frac` | (P,P) | Örneklenen aksiyon |
| `old_target_logp` | (P,) | PPO ratio için |
| `target_valid` | (P,) | PPO maskesi |
| `credit_target` | (P,) | Episode sonu kampanya credit |
| `adv_target` | (P,) | `credit_target` (GAE sonrası) |
| `reward`, `value` | skaler | Value head GAE |

Kampanya credit → `(record_idx, t_local)` ile **launch turuna** yazılır (`buffer.edge_decisions`).

## 9) Ödül ve öğrenme (episode sonu)

### Remastered kampanya credit
- **Attack:** hedef ele geçince `+dpp × (1 − casualty − delay)`
- **Fail:** yeterli güçle alınamazsa `−w_fail × …`
- **Defense loss:** gezegen kaybı

Credit launch payına göre `(record_idx, target_slot)`'a dağıtılır.

### GAE (güncel)
```
adv_target[k] = credit_target[k]   if target_valid[k] else 0
```
Tur-level `tr.advantage` yalnızca **value head** için.

### PPO (per-target)
```
loss = − mean_over_valid_targets( min(r·A, clip(r)·A) )
denom = target_valid.sum()
```

Bu yüzden `target_valid`'i daraltmak (gate açık / launch) kampanya sinyalini güçlendirir.

## 10) Özet tablo — veri boyutları

| Aşama | Çıktı | Boyut |
|-------|-------|-------|
| Ham planet | liste | 7 sayı / gezegen |
| obs_builder | planet_feats | (44, 34) |
| reworked node | node_feats | (n, 16) |
| reworked edge | edge_feats | (n, n, 4) |
| Encoder | node_tokens | (B, P, d_model) |
| Policy | gate, frac | (B, P, P) |
| Env | action | list of [id, θ, ships] |
| Buffer | adv_target | (P,) per turn |

## 11) İlgili dosyalar

| Dosya | Rol |
|-------|-----|
| `obs_builder.py` | Ham dict → Observation |
| `reworked_pipeline.py` | Observation → BoardState + features |
| `reworked_features.py` | dpp, threat, affordability, flux … |
| `reworked_model.py` | GNN + ReworkedEdgePolicy |
| `reworked_rollout.py` | mask, env launch, buffer, GAE, PPO |
| `reworked_remastered_reward.py` | Kampanya credit |
| `navigation.py` | `compute_launch` (açı, ETA) |
| `reworked_watch.py` | Replay (deterministic act) |